# Sampling & Estimation (CFA Level 1)
## From Sampling Methods to Confidence Intervals and Bootstrap Inference

This notebook provides a rigorous, from-scratch treatment of **sampling** and **estimation** theory central to the CFA Level 1 curriculum.

**What you will learn:**
1. Sampling methods: simple random, stratified, systematic
2. The Central Limit Theorem (CLT) and its proof by simulation
3. Point estimators and their desirable properties
4. Confidence intervals (z and t) from scratch
5. Bootstrap methods for nonparametric inference
6. Sample size determination and power analysis

**Why it matters:** Financial analysts never observe the true population -- they work with samples. Understanding how to draw valid inferences from limited data, and how to quantify uncertainty, is essential for sound investment decisions.

**Prerequisites:** Basic probability, normal distribution.

**References:**
- CFA Institute, *CFA Program Curriculum*, Quantitative Methods.
- DeFusco, R. et al., *Quantitative Investment Analysis*, CFA Institute, Wiley.

---

## Why Sampling Matters in Finance

In finance, we almost never observe the entire population. We don't know the "true" expected return of a stock — we only have a sample of historical returns. We don't know every investor's opinion — we survey a sample.

**Sampling and estimation** give us the tools to make rigorous inferences about populations from samples, and to quantify our uncertainty.

| Concept | Financial Application |
|:--------|:---------------------|
| Sampling distribution | How much will my estimate of expected return vary from sample to sample? |
| Central Limit Theorem | Why can we use normal-distribution formulas even when returns aren't normal? |
| Confidence intervals | What range of values is the true Sharpe ratio likely to be in? |
| Bootstrap | How do I estimate uncertainty for complex statistics (median, VaR) with no formula? |
| Sample size | How many months of data do I need for a reliable beta estimate? |

> **Key Concept:** The **Central Limit Theorem (CLT)** is arguably the most important theorem in statistics. It says: no matter what the underlying distribution looks like, the distribution of the sample mean approaches a normal distribution as sample size grows. This is why the normal distribution appears everywhere in finance.

### The Big Picture: Population vs. Sample

Before diving in, let us make sure the foundational vocabulary is crystal clear:

| Term | Symbol | Meaning | Example |
|:-----|:-------|:--------|:--------|
| **Population** | — | The complete set of all items of interest | All monthly returns of the S&P 500 since inception |
| **Sample** | — | A subset drawn from the population | The last 60 monthly returns |
| **Parameter** | $\mu$, $\sigma$ | A fixed (but usually unknown) number describing the population | The true long-run average monthly return |
| **Statistic** | $\bar{X}$, $s$ | A number computed from the sample, used to *estimate* the parameter | The average return in our 60-month window |
| **Sampling distribution** | — | The probability distribution of a statistic across all possible samples of a given size | The distribution of $\bar{X}$ if we could draw every possible 60-month window |
| **Standard error** | $SE(\bar{X})$ | The standard deviation of the sampling distribution | How much $\bar{X}$ typically varies from sample to sample |

The entire chapter rests on one core idea: **a statistic is itself a random variable**. Every time you draw a different sample, you get a different value of $\bar{X}$. The sampling distribution describes *how much* it fluctuates, and the CLT tells us *what shape* that fluctuation takes.

> **CFA Exam Tip:** The exam loves to test whether you can distinguish between a parameter and a statistic. Remember: parameters are *fixed but unknown*; statistics are *known but random*. We use statistics to estimate parameters.

### A Road Map for This Notebook

We will proceed through the material in a logical sequence:

1. **Sampling methods** — how do we draw a representative sample? (Simple random, stratified, systematic)
2. **The CLT** — once we have a sample, what can we say about the distribution of its mean?
3. **Point estimators** — which formulas give us "good" estimates, and what does "good" mean precisely?
4. **Confidence intervals** — how do we put an uncertainty band around our point estimate?
5. **Bootstrap** — what if the statistic is too complicated for a formula? Simulate!
6. **Sample size determination** — before collecting data, how much do we need?

Each section pairs theory with a from-scratch NumPy implementation and a visualization.

### Setup

Let's set up our environment. We'll use NumPy for calculations and SciPy for statistical functions.

We set a random seed for reproducibility. This means if you run this notebook yourself, you will get exactly the same random samples and the same results. In practice, you would not fix the seed — but for a pedagogical notebook, reproducibility is essential.

The plot style constants (`PRIMARY`, `SECONDARY`, etc.) are just color choices to keep the visualizations consistent throughout.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ── Tolerances
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 1. Sampling Methods

Before we can estimate anything, we need data — and *how* we collect that data matters enormously. A biased sample leads to biased conclusions, no matter how sophisticated our estimator is. The CFA curriculum identifies three primary probability sampling methods.

### Simple Random Sampling

Each member of the population has an **equal probability** of being selected. This is the gold standard and the method assumed by most statistical formulas.

**How it works:** Imagine writing every stock ticker in the S&P 500 on a slip of paper, putting them in a hat, and blindly drawing 50 slips. Each stock had the same $50/500 = 10\%$ chance of being selected.

**Strengths:**
- Unbiased by construction — no systematic over- or under-representation
- All standard formulas (CLT, confidence intervals) are directly applicable

**Weaknesses:**
- May fail to capture small but important subgroups (e.g., if only 3% of the population is in energy, your random sample of 50 might contain zero energy stocks)
- Can be expensive when the population is large and hard to enumerate

### Stratified Random Sampling

Divide the population into non-overlapping **strata** (subgroups) and sample proportionally from each. This guarantees representation from every stratum.

**How it works:** Divide the S&P 500 into sectors (Tech, Healthcare, Utilities, etc.). If Tech is 40% of the index, draw 40% of your sample from Tech. Within each sector, use simple random sampling.

**Strengths:**
- Reduces sampling error when strata are internally homogeneous but differ from each other
- Guarantees representation of every subgroup
- In finance, this is how index fund managers build "representative" portfolios — they stratify by sector, market cap, and region

**Weaknesses:**
- Requires knowing the strata in advance
- More complex to implement

> **Key Concept:** Stratified sampling is almost always more precise than simple random sampling (i.e., lower standard error for the same total sample size). The intuition is straightforward: by forcing representation from each stratum, you eliminate one source of random variation — whether a particular stratum happens to be over- or under-represented.

### Systematic Sampling

Select every $k$-th element from an ordered list, starting from a random point. For example, if you have 2,000 stocks and want 100, pick a random starting point between 1 and 20, then take every 20th stock.

**How it works:**
1. Compute the sampling interval: $k = N/n$ (population size / sample size)
2. Pick a random start between 1 and $k$
3. Select items at positions: start, start + $k$, start + $2k$, ...

**Strengths:**
- Extremely simple to implement
- Spreads the sample evenly across the list

**Weaknesses:**
- Can introduce bias if the list has a periodic pattern (e.g., if every 20th stock is a bank because the list is sorted by sector)
- Not truly random — once you pick the start, the rest is determined

> **CFA Exam Tip:** The exam may describe a scenario and ask you to identify which sampling method is being used. The giveaway for systematic sampling is the phrase "every $k$-th element" or "sampling interval." For stratified, look for "divided into subgroups" or "proportional allocation."

### Sampling Bias: What Can Go Wrong

Even with the right sampling *method*, financial data introduces several biases:

- **Survivorship bias:** If your database only contains stocks that still exist today, you are systematically excluding companies that went bankrupt — making average returns look better than they truly were.
- **Look-ahead bias:** Using information that would not have been available at the time of the investment decision (e.g., using full-year earnings to make a January trading decision).
- **Time-period bias:** Results that depend heavily on the specific dates chosen (a study of stock returns starting in March 2009 looks very different from one starting in October 2007).

> **Common Mistake:** Many students focus on the mathematical formulas and forget about sampling bias. On the CFA exam, if a question describes a study with obvious survivorship bias, the correct answer is often that the *results are unreliable* — no amount of statistical sophistication can fix a biased sample.

### Implementing Sampling Methods

Let's create a simulated "population" of stock returns and draw samples using each method. We'll compare how well each method estimates the true population mean.

Our population consists of 2,000 stocks from three sectors with different return characteristics:

| Sector | Count | Mean Return | Volatility | Character |
|:-------|------:|:-----------:|:----------:|:----------|
| Tech | 800 | 15% | 30% | High return, high risk |
| Healthcare | 700 | 10% | 20% | Moderate |
| Utilities | 500 | 6% | 10% | Low return, low risk |

We will draw 1,000 samples of size $n=100$ using each method and compare how precisely each estimates the true population mean.

For stratified sampling, we use **proportional allocation**: each sector's share in the sample matches its share in the population (Tech: 40%, Healthcare: 35%, Utilities: 25%).

The comparison metrics are:
- **Mean of means**: should equal the true population mean if the method is unbiased
- **Std of means**: the empirical standard error — lower means more precise
- **MSE**: Mean Squared Error = Bias$^2$ + Variance — the single best measure of estimator quality

> **What to watch for:** Stratified sampling should give the most precise estimates because it ensures representation from all sectors. Simple random sampling might occasionally over-sample Tech (high variance), leading to wider swings in the estimated mean. Systematic sampling depends on list ordering.

In [ ]:
def simple_random_sample(population, n, rng):
    """Draw n items from population without replacement."""
    indices = rng.choice(len(population), size=n, replace=False)
    return population[indices]


def stratified_sample(population, strata_labels, n_per_stratum, rng):
    """Stratified random sampling.
    
    Parameters
    ----------
    population     : array of values
    strata_labels  : array of stratum identifiers (same length)
    n_per_stratum  : dict mapping stratum label -> sample size
    """
    sample = []
    for stratum, n_s in n_per_stratum.items():
        mask = strata_labels == stratum
        stratum_pop = population[mask]
        indices = rng.choice(len(stratum_pop), size=n_s, replace=False)
        sample.extend(stratum_pop[indices])
    return np.array(sample)


def systematic_sample(population, n, rng):
    """Systematic sampling: every k-th element."""
    N = len(population)
    k = N // n  # sampling interval
    start = rng.integers(0, k)
    indices = np.arange(start, N, k)[:n]
    return population[indices]


# ── Create a population: stock returns by sector
N_pop = 2000

# Three sectors with different return distributions
tech_returns = rng.normal(0.15, 0.30, 800)      # high return, high vol
healthcare_returns = rng.normal(0.10, 0.20, 700) # moderate
utilities_returns = rng.normal(0.06, 0.10, 500)  # low return, low vol

population = np.concatenate([tech_returns, healthcare_returns, utilities_returns])
strata = np.array(['Tech'] * 800 + ['Healthcare'] * 700 + ['Utilities'] * 500)

# Population parameter (what we're trying to estimate)
pop_mean = np.mean(population)
print(f"Population size: {len(population)}")
print(f"True population mean: {pop_mean:.4%}")

# ── Compare sampling methods (n=100)
n_sample = 100
n_trials = 1000

means_srs = []
means_strat = []
means_sys = []

for _ in range(n_trials):
    # Simple random
    s = simple_random_sample(population, n_sample, rng)
    means_srs.append(np.mean(s))
    
    # Stratified (proportional allocation)
    s = stratified_sample(population, strata, 
                          {'Tech': 40, 'Healthcare': 35, 'Utilities': 25}, rng)
    means_strat.append(np.mean(s))
    
    # Systematic
    s = systematic_sample(population, n_sample, rng)
    means_sys.append(np.mean(s))

print(f"\n{'Method':<20} {'Mean of means':>14} {'Std of means':>14} {'MSE':>14}")
print("-" * 64)
for name, means in [('Simple Random', means_srs), 
                     ('Stratified', means_strat),
                     ('Systematic', means_sys)]:
    m = np.mean(means)
    s = np.std(means)
    mse = np.mean((np.array(means) - pop_mean) ** 2)
    print(f"{name:<20} {m:>14.4%} {s:>14.4%} {mse:>14.8f}")

### Interpreting the Sampling Method Comparison

Let us unpack what these numbers are telling us.

**Mean of means** — all three methods should produce means very close to the true population mean. This is because all three are *probability* sampling methods (as opposed to convenience sampling), so they are unbiased. If any method's "mean of means" differed substantially from the true population mean, that would signal a systematic bias.

**Std of means** — this is the empirical *standard error*. It measures how much the sample mean bounces around from trial to trial. Lower is better. You should see that:

- **Stratified** has the lowest standard error — it eliminates between-sector variability by design
- **Simple random** and **systematic** are comparable, with simple random typically a bit higher

**MSE (Mean Squared Error)** — this combines both bias and variance: $MSE = Bias^2 + Variance$. Since all methods are approximately unbiased here, MSE essentially tracks variance.

> **Key Concept: The Bias-Variance Decomposition.** For any estimator $\hat{\theta}$:
> $$MSE(\hat{\theta}) = [Bias(\hat{\theta})]^2 + Var(\hat{\theta})$$
> A low MSE requires *both* low bias and low variance. An estimator can have zero bias but terrible MSE (if its variance is huge), or low variance but terrible MSE (if it is consistently wrong in the same direction).

**Why stratified wins.** Think of it this way. In simple random sampling, you *might* draw 60 Tech stocks and only 10 Utilities by chance. Tech has much higher variance, so your mean gets pulled around. Stratified sampling *forces* exactly 40 Tech, 35 Healthcare, and 25 Utilities every time — eliminating that source of randomness.

**A practical illustration:** Suppose a portfolio manager wants to build a 100-stock portfolio that tracks the full 2,000-stock universe. With simple random sampling, the portfolio might accidentally overweight Tech in one quarter and underweight it the next — creating tracking error. With stratified sampling, the sector weights are locked in, and the only source of tracking error is *within-sector* stock selection. This is why real index funds use stratified (or "optimised") sampling.

> **CFA Exam Tip:** When a question asks which sampling method is "most efficient" or "most precise" for a population with distinct subgroups, the answer is almost always stratified random sampling.

### Visualising the Sampling Distributions

The histograms below show the *sampling distribution* of $\bar{X}$ under each method. Notice that all three are centered on the true mean (unbiased), but their spreads differ. The tighter the histogram, the more precise the method.

Also note that all three histograms look approximately normal — this is the CLT at work, which we will explore in depth in the next section.

In [ ]:
# ── Visualization: Sampling distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, means, name, color in zip(axes,
    [means_srs, means_strat, means_sys],
    ['Simple Random', 'Stratified', 'Systematic'],
    [PRIMARY, SECONDARY, TERTIARY]):
    
    ax.hist(means, bins=40, density=True, alpha=0.7, color=color, edgecolor='white')
    ax.axvline(pop_mean, color='black', linestyle='--', linewidth=2, label=f'True μ = {pop_mean:.2%}')
    ax.axvline(np.mean(means), color='darkred', linestyle=':', linewidth=2, 
               label=f'Sample mean = {np.mean(means):.2%}')
    ax.set_title(f'{name}\nσ(x̄) = {np.std(means):.4f}')
    ax.set_xlabel('Sample Mean')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Density')
plt.suptitle('Comparison of Sampling Methods (n=100, 1000 trials)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Reading the Histograms

Each histogram represents what would happen if an analyst repeated their study 1,000 times, each time drawing a fresh sample of 100 stocks.

- The **black dashed line** is the true population mean — the target we are trying to estimate.
- The **red dotted line** is the average of all 1,000 sample means. It should sit very close to the true mean for all three methods (confirming unbiasedness).
- The **width** of each histogram reflects the standard error. Narrower is better.

Notice how the stratified histogram is noticeably narrower. In a real-world context, this means a portfolio manager using stratified sampling to build an index-tracking portfolio will get closer to the true index characteristics with fewer stocks.

> **Common Mistake:** Students sometimes confuse the sampling distribution (distribution of a *statistic* across many samples) with the population distribution (distribution of individual observations). The sampling distribution is always narrower than the population distribution — by a factor of $\sqrt{n}$.

### Why the Sampling Distribution Matters

The sampling distribution is arguably the most important concept in inferential statistics. Here is why:

- Every **confidence interval** you compute is based on the sampling distribution.
- Every **hypothesis test** you run asks whether the observed statistic is unusual *relative to its sampling distribution under $H_0$*.
- Every **p-value** is a probability calculated from the sampling distribution.

If you do not understand the sampling distribution, the rest of the chapter will feel like memorised formulas without meaning. If you *do* understand it, everything else follows naturally.

> **Key Concept:** The sampling distribution answers the question: "If the true parameter is $\theta$ and I draw a sample of size $n$, what values of my statistic are likely?" It is a *thought experiment* — we imagine drawing many samples — but the CLT lets us characterise it mathematically without actually repeating the experiment.

---
## 2. Central Limit Theorem (CLT)

The Central Limit Theorem is, without exaggeration, **the most important theorem in all of applied statistics**. It is the reason we can use normal-distribution formulas for hypothesis tests and confidence intervals even when the underlying data is not normally distributed.

### The Formal Statement

Let $X_1, X_2, \ldots, X_n$ be i.i.d. random variables with mean $\mu$ and finite variance $\sigma^2$. Then as $n \to \infty$:

$$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} N(0, 1)$$

Equivalently: $\bar{X}_n \approx N\left(\mu, \frac{\sigma^2}{n}\right)$ for large $n$.

### In Plain English

If you take a large enough sample, the *average* of that sample will be approximately normally distributed — **no matter what the individual observations look like**. They could be skewed, bimodal, uniform, exponential — it does not matter. Average enough of them together and the result is normal.

### Why Does It Work? An Intuitive Explanation

The deep reason the CLT works is *cancellation of extremes*. Here is the intuition:

1. **Individual observations are unpredictable.** A single stock return could be anywhere — a crash, a boom, or somewhere in between.

2. **Averages smooth things out.** When you average many observations, the extreme highs and lows tend to cancel each other. A few very high returns are offset by a few very low returns.

3. **The more you average, the more cancellation occurs.** With $n = 5$, cancellation is partial — the average is still somewhat unpredictable. With $n = 100$, cancellation is nearly complete — the average is tightly clustered around $\mu$.

4. **The shape of the clustering is always normal.** This is the magical part. The *rate* at which cancellation occurs follows a precise mathematical pattern that produces a bell curve, regardless of the original distribution.

A more mathematical way to see it: the *moment-generating function* (or equivalently, the characteristic function) of the standardized sum converges to that of a standard normal. Each additional observation contributes a multiplicative factor that, in the limit, produces the Gaussian MGF $e^{t^2/2}$.

### The Three Key Implications

1. **Normality of the sample mean.** The sampling distribution of $\bar{X}$ is approximately normal **regardless of the population distribution** — provided $n$ is large enough.

2. **Standard error shrinks with $\sqrt{n}$.** The standard deviation of $\bar{X}$ is $\sigma/\sqrt{n}$. This means:
   - Doubling precision (halving the standard error) requires **4x** the data
   - Getting 10x more precise requires **100x** the data
   - There are diminishing returns to collecting more observations

3. **The magic number $n \geq 30$.** For most distributions, $n \geq 30$ is sufficient for the CLT approximation to be reasonably accurate. For heavily skewed distributions, you may need $n \geq 50$ or more.

> **Key Concept: The $\sqrt{n}$ Law.** The standard error of the mean is $\sigma / \sqrt{n}$. This is one of the most important formulas in statistics. It tells you exactly how uncertainty in your estimate decreases as you collect more data. Want to cut your estimation error in half? You need four times as much data.

### Why It Matters for Finance

Stock returns are not normally distributed. They exhibit:
- **Negative skewness** — large losses are more common than large gains
- **Fat tails (excess kurtosis)** — extreme events occur more often than a normal distribution predicts
- **Time-varying volatility** — calm periods alternate with crisis periods

Despite all this, the CLT guarantees that *sample means* of returns are approximately normal. This is why z-tests and t-tests work for testing hypotheses about expected returns. It is also why portfolio theory (which relies on means and covariances) is reasonably well-grounded even though individual returns are non-normal.

> **Common Mistake:** The CLT says the *sample mean* is approximately normal. It does NOT say that the *individual observations* become normal as $n$ grows. The population distribution stays whatever it is. It is the distribution of the *average* that becomes normal.

### Worked Example: CLT in Action

Suppose daily returns on a volatile stock have mean $\mu = 0.04\%$ and standard deviation $\sigma = 2\%$, and the distribution is heavily right-skewed (not normal at all).

**Question:** What is the approximate distribution of the average daily return computed from a sample of $n = 50$ trading days?

**Answer:** By the CLT:
$$\bar{X}_{50} \approx N\left(0.04\%,\ \left(\frac{2\%}{\sqrt{50}}\right)^2\right) = N(0.04\%,\ (0.283\%)^2)$$

So the average daily return over 50 days is approximately normal with mean 0.04% and standard error 0.283%. The skewness of the original distribution is irrelevant for the *average* — the CLT has "washed it out."### Why the CLT Is Revolutionary

Consider what the CLT does NOT require:
- It does NOT require the population to be normal
- It does NOT require knowing the population distribution at all
- It works for skewed, bimodal, uniform, or any other distribution

As long as the population has a finite mean and variance, the CLT kicks in. This is why virtually every hypothesis test and confidence interval in statistics works — they all rely on the sampling distribution of the mean being approximately normal.

> **CFA Exam Tip:** The CLT applies to the distribution of the **sample mean**, NOT to the distribution of individual observations. A common exam trap is confusing these two. Individual stock returns may be skewed, but the average of 30+ monthly returns is approximately normal.


### Demonstrating the CLT with Simulation

The best way to truly understand the CLT is to see it in action. We'll take a population with a clearly non-normal distribution (exponential — heavily right-skewed) and show that sample means become normal anyway.

The exponential distribution with rate $\lambda = 2$ has:
- Mean: $\mu = 1/\lambda = 0.5$
- Std: $\sigma = 1/\lambda = 0.5$
- Skewness: 2 (always positive, always right-skewed)

It looks nothing like a bell curve — it is a sharply declining curve starting from zero, with a long right tail.

We will draw 10,000 samples of sizes $n = 1, 2, 5, 10, 30, 100$ and plot the histogram of the *standardized* sample mean:

$$Z = \frac{\bar{X} - \mu}{\sigma / \sqrt{n}}$$

If the CLT is working, this should converge to a standard normal $N(0,1)$ as $n$ increases.

> **What to expect:** At $n=1$, the histogram looks purely exponential (because the "mean" of one observation is just that observation). By $n=5$, it is already less skewed. By $n=30$, it looks remarkably bell-shaped. By $n=100$, it is nearly indistinguishable from the $N(0,1)$ overlay.

In [ ]:
# ── CLT demonstration: sample mean converges to normal
# Start with a VERY non-normal population: exponential distribution

pop_lambda = 2.0  # rate parameter
pop_mu = 1 / pop_lambda  # theoretical mean
pop_sigma = 1 / pop_lambda  # theoretical std

sample_sizes = [1, 2, 5, 10, 30, 100]
n_sims = 10_000

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, n in enumerate(sample_sizes):
    # Draw n_sims samples, each of size n
    sample_means = np.array([np.mean(rng.exponential(1/pop_lambda, n)) for _ in range(n_sims)])
    
    # Standardize
    z_scores = (sample_means - pop_mu) / (pop_sigma / np.sqrt(n))
    
    ax = axes[idx]
    ax.hist(z_scores, bins=60, density=True, alpha=0.7, color=PRIMARY, edgecolor='white')
    
    # Overlay standard normal
    x_range = np.linspace(-4, 4, 200)
    ax.plot(x_range, stats.norm.pdf(x_range), color=SECONDARY, linewidth=2, label='N(0,1)')
    
    ax.set_title(f'n = {n}')
    ax.set_xlim(-4, 4)
    ax.legend()

plt.suptitle('Central Limit Theorem: Exponential Population → Normal Sample Means',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("As n increases, the sampling distribution of the standardized mean")
print("converges to N(0,1), even though the population is exponential (highly skewed).")

### What the CLT Simulation Tells Us

Look carefully at the six panels:

1. **$n = 1$:** The histogram is the exponential distribution itself (right-skewed, bounded at zero on the left). The $N(0,1)$ curve is a poor fit.

2. **$n = 2$:** Already less skewed — averaging just two observations provides partial cancellation of extremes. But still clearly non-normal.

3. **$n = 5$:** The distribution is starting to look symmetric and bell-shaped, though the right tail is still a bit heavier.

4. **$n = 10$:** Very close to normal. A formal normality test might still reject, but the visual agreement is striking.

5. **$n = 30$:** The histogram tracks the $N(0,1)$ curve almost perfectly. This is why textbooks cite $n \geq 30$ as a rule of thumb.

6. **$n = 100$:** Virtually indistinguishable from normal.

This is the CLT in action. **We started with an exponential distribution (skewness = 2) and ended up with a perfect bell curve** — simply by averaging.

> **Key Concept:** The speed of CLT convergence depends on the population's skewness. Symmetric distributions (like uniform) converge very fast — even $n = 5$ looks normal. Highly skewed distributions (like exponential or Pareto) need $n \geq 30$ or more. The CFA curriculum typically uses $n \geq 30$ as the threshold.

### A Financial Analogy

Think of individual stock returns as individual exponential draws — highly variable, asymmetric, hard to predict. Now think of a **diversified portfolio** as an *average* of those individual returns. The CLT is telling us that the *portfolio return* (a weighted average of many stocks) will be much more "normal-looking" than any individual stock return. This is the statistical foundation of diversification.

> **CFA Exam Tip:** If an exam question gives you a non-normal population and asks about the distribution of the sample mean with $n = 50$, you can confidently invoke the CLT and say the sample mean is approximately normally distributed. The CLT is your license to use normal-based formulas.

### The Standard Error: Quantifying Estimation Uncertainty

The CLT tells us *shape* (normal). But we also need to know *spread* — how much does $\bar{X}$ vary from sample to sample? This is captured by the **standard error**:

$$SE(\bar{X}) = \frac{\sigma}{\sqrt{n}}$$

This formula encodes a profound insight: **estimation precision grows with the square root of sample size, not linearly**.

| Sample size $n$ | Standard error (as fraction of $\sigma$) | Relative improvement |
|:---------------:|:---------------------------------------:|:--------------------:|
| 1 | $\sigma$ | baseline |
| 4 | $\sigma / 2$ | 2x better |
| 25 | $\sigma / 5$ | 5x better |
| 100 | $\sigma / 10$ | 10x better |
| 400 | $\sigma / 20$ | 20x better |
| 10,000 | $\sigma / 100$ | 100x better |

To go from "10x better" to "20x better," you need to increase $n$ from 100 to 400 — not from 100 to 200. This is the **law of diminishing returns** for data collection.

> **Key Concept:** In practice, this means a fund with 10 years of monthly data ($n = 120$) has a standard error that is only $\sqrt{120/60} = 1.41$ times smaller than a fund with 5 years of data ($n = 60$). Getting meaningfully more precision requires dramatically more data — which is why long track records are so valuable in finance.

The code below verifies this theoretical relationship empirically. We plot $\sigma / \sqrt{n}$ alongside actual empirical standard errors computed from simulations.> **Key Concept:** The standard error of the mean decreases with the **square root** of sample size:
> $$SE = \frac{\sigma}{\sqrt{n}}$$
> This square-root relationship has a profound practical implication: to halve the standard error, you need **four times** as many observations. Going from $n = 100$ to $n = 400$ halves the error; going from 400 to 1600 halves it again. Precision gets expensive fast.


In [ ]:
# ── Standard error decreases with sqrt(n)
n_values = np.arange(5, 501)
theoretical_se = pop_sigma / np.sqrt(n_values)

# Verify empirically
empirical_se = []
check_n_values = [5, 10, 25, 50, 100, 200, 500]
for n in check_n_values:
    means = [np.mean(rng.exponential(1/pop_lambda, n)) for _ in range(5000)]
    empirical_se.append(np.std(means))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_values, theoretical_se, color=PRIMARY, linewidth=2, label=r'$\sigma/\sqrt{n}$ (theoretical)')
ax.scatter(check_n_values, empirical_se, color=SECONDARY, s=80, zorder=5, label='Empirical SE')
ax.set_xlabel('Sample Size (n)')
ax.set_ylabel('Standard Error of Mean')
ax.set_title('Standard Error Decreases with Sample Size')
ax.legend()
plt.tight_layout()
plt.show()

### Interpreting the Standard Error Plot

The blue curve is the theoretical formula $\sigma / \sqrt{n}$, and the orange dots are the actual standard errors computed from 5,000 simulations at each sample size. The agreement is excellent, confirming that the CLT formula is accurate.

Notice the shape of the curve: it drops steeply at first (going from $n=5$ to $n=50$ buys a lot of precision) but then flattens out (going from $n=200$ to $n=500$ provides diminishing returns). This has important practical implications:

- **For a new fund** with only 12 months of data, every additional month provides substantial improvement.
- **For an established fund** with 20 years (240 months) of data, adding another year barely moves the needle.

### The Practical Cost of Precision

Let us put concrete numbers on the diminishing returns:

| From $n$ | To $n$ | Extra data needed | Improvement in SE |
|:--------:|:------:|:-----------------:|:-----------------:|
| 10 | 40 | 30 more observations | 2x better |
| 40 | 160 | 120 more observations | 2x better |
| 160 | 640 | 480 more observations | 2x better |

Each doubling of precision costs 4x as much data. In financial terms, if your fund has been running for 5 years and you want to cut the confidence interval width in half, you need to wait *15 more years* (for a total of 20 years) — not just another 5 years.

This is the fundamental reason why financial research is plagued by wide confidence intervals and low statistical power. The returns are noisy, the signal is small, and the data accumulates slowly (one month at a time).

> **Common Mistake:** Some analysts assume that "more data is always better" without considering the $\sqrt{n}$ law. In reality, doubling your data only reduces your standard error by about 30% (factor of $\sqrt{2} \approx 1.41$). If your confidence interval is already too wide, you may need *dramatically* more data — or a better estimation method (like stratified sampling).

---
## 3. Point Estimators

A **point estimator** $\hat{\theta}$ is a single number computed from the sample, used to estimate a population parameter $\theta$. The sample mean $\bar{X}$ is a point estimator for the population mean $\mu$. The sample variance $s^2$ is a point estimator for the population variance $\sigma^2$.

But not all estimators are created equal. How do we judge whether an estimator is "good"? The CFA curriculum identifies three desirable properties.

### Property 1: Unbiasedness

An estimator is **unbiased** if its expected value equals the true parameter:

$$E[\hat{\theta}] = \theta$$

**Intuition:** If you repeated the sampling process infinitely many times and averaged all the estimates, you would get exactly the true value.

**Example — Sample Mean:** The sample mean $\bar{X} = \frac{1}{n}\sum_{i=1}^n X_i$ is unbiased for $\mu$ because:
$$E[\bar{X}] = E\left[\frac{1}{n}\sum X_i\right] = \frac{1}{n}\sum E[X_i] = \frac{1}{n} \cdot n\mu = \mu$$

**Example — Sample Variance:** The formula $s^2 = \frac{1}{n-1}\sum(X_i - \bar{X})^2$ uses $n-1$ (not $n$) in the denominator. This is called **Bessel's correction** and it makes the estimator unbiased. Using $n$ in the denominator (as in the MLE) gives a **biased** estimator that systematically underestimates $\sigma^2$.

> **Key Concept: Why $n-1$ instead of $n$?** When you compute $\sum(X_i - \bar{X})^2$, you are measuring deviations from the *sample mean* rather than the *true mean*. Since the sample mean is computed from the same data, the deviations are systematically smaller than they would be relative to $\mu$. Dividing by $n-1$ corrects for this "lost degree of freedom." Think of it this way: if you know $\bar{X}$ and $n-1$ of the observations, you can deduce the last observation — so you really only have $n-1$ independent pieces of information about spread.

### Property 2: Efficiency

Among all unbiased estimators of $\theta$, the **efficient** one has the smallest variance.

**Intuition:** Two thermometers might both be unbiased (correct on average), but you prefer the one with less random fluctuation. An efficient estimator is the most "stable" unbiased estimator.

**Example:** For a normal population, $\bar{X}$ is more efficient than the sample median as an estimator of $\mu$. The sample median is also unbiased, but it has about 57% more variance than $\bar{X}$ for normal data. (However, the median is more robust to outliers, so efficiency is not everything.)

The **Cramer-Rao Lower Bound** gives the theoretical minimum variance for any unbiased estimator. An estimator that achieves this bound is called **MVUE** (Minimum Variance Unbiased Estimator).

### Property 3: Consistency

An estimator is **consistent** if it converges to the true parameter as sample size grows:

$$\hat{\theta}_n \xrightarrow{p} \theta \text{ as } n \to \infty$$

**Intuition:** With more data, the estimate gets closer and closer to the truth. There is no guarantee it will be exactly right with any finite sample, but it will eventually get arbitrarily close.

**Example:** Both $\bar{X}$ and $s^2$ are consistent estimators. Even the biased MLE variance estimator $\hat{\sigma}^2_{MLE} = \frac{1}{n}\sum(X_i - \bar{X})^2$ is consistent — its bias is $-\sigma^2/n$, which vanishes as $n \to \infty$.

> **CFA Exam Tip:** A sufficient condition for consistency: if an estimator is unbiased and its variance shrinks to zero as $n \to \infty$, then it is consistent. The sample mean satisfies this: it is unbiased and $Var(\bar{X}) = \sigma^2/n \to 0$.

### Summary of Estimator Properties

| Property | What it means | Formula / Condition | Analogy |
|:---------|:-------------|:-------------------|:--------|
| Unbiased | Correct on average | $E[\hat{\theta}] = \theta$ | A scale that is right *on average* (even if readings fluctuate) |
| Efficient | Least spread | Smallest $Var(\hat{\theta})$ among unbiased estimators | The most *precise* scale |
| Consistent | Improves with data | $\hat{\theta}_n \to \theta$ as $n \to \infty$ | Readings converge to truth as you average more |

### Maximum Likelihood Estimation (MLE)

Given data $x_1, \ldots, x_n$ from a distribution $f(x|\theta)$, the MLE maximizes the **likelihood function** — the probability of observing the data as a function of the parameter:

$$\hat{\theta}_{\text{MLE}} = \arg\max_{\theta} \prod_{i=1}^{n} f(x_i|\theta) = \arg\max_{\theta} \sum_{i=1}^{n} \ln f(x_i|\theta)$$

We work with the log-likelihood (the sum) because it is numerically more stable and easier to differentiate.

**For the normal distribution**, the MLE gives:
- $\hat{\mu}_{MLE} = \bar{X}$ (same as the sample mean — unbiased!)
- $\hat{\sigma}^2_{MLE} = \frac{1}{n}\sum(X_i - \bar{X})^2$ (biased — uses $n$ not $n-1$)

The MLE for variance is biased, but it is consistent and asymptotically efficient. In practice, for $n > 30$ the difference between dividing by $n$ and $n-1$ is negligible.

> **Common Mistake:** Confusing MLE variance (divides by $n$) with sample variance (divides by $n-1$). The CFA exam typically uses $n-1$ (the unbiased version). The MLE version appears in some statistics courses but not usually in the CFA curriculum. Know both, but default to $n-1$ unless told otherwise.### Desirable Properties of Estimators

| Property | Meaning | Example |
|:---------|:--------|:--------|
| **Unbiased** | On average, the estimate equals the true parameter: $E[\hat{\theta}] = \theta$ | Sample mean is unbiased for $\mu$ |
| **Efficient** | Has the smallest variance among all unbiased estimators | Sample mean is more efficient than sample median for normal data |
| **Consistent** | Converges to the true parameter as $n \to \infty$ | Both mean and median are consistent |

> **CFA Exam Tip:** The sample variance with $n-1$ in the denominator ($s^2 = \frac{1}{n-1}\sum(x_i - \bar{x})^2$) is unbiased for $\sigma^2$. With $n$ in the denominator, it's biased (the MLE). The CFA curriculum uses $n-1$ (Bessel's correction).


### Demonstrating Unbiasedness and Consistency

The code below runs a Monte Carlo experiment to verify that $\bar{X}$ and $s^2$ are unbiased estimators. We draw 5,000 samples from a $N(0.10, 0.20^2)$ distribution at various sample sizes and check:

1. **Unbiasedness:** Is the average of all 5,000 estimates close to the true parameter? (The "Bias" columns should be near zero.)
2. **Consistency:** Does the bias get even smaller as $n$ increases? (It should — though for truly unbiased estimators, the bias is zero at every $n$, not just in the limit.)

We also implement the MLE for a normal distribution and compare.

> **Key Concept:** The sample mean $\bar{X}$ is the MLE for the population mean $\mu$. The MLE for variance uses $N$ in the denominator (biased), while the unbiased estimator uses $N-1$.

In [ ]:
# ── Demonstration: Unbiasedness and Consistency

true_mu = 0.10
true_sigma = 0.20
n_sims = 5000

# Check unbiasedness for different sample sizes
print(f"True parameters: μ = {true_mu}, σ² = {true_sigma**2:.4f}\n")
print(f"{'n':>6} {'E[x̄] (μ̂)':>12} {'E[s²] (σ̂²)':>14} {'Bias(x̄)':>12} {'Bias(s²)':>12}")
print("-" * 58)

sample_sizes_check = [5, 10, 30, 100, 500]
for n in sample_sizes_check:
    means = []
    variances = []
    for _ in range(n_sims):
        sample = rng.normal(true_mu, true_sigma, n)
        means.append(np.mean(sample))
        variances.append(np.sum((sample - np.mean(sample))**2) / (n - 1))  # Bessel's
    
    e_mean = np.mean(means)
    e_var = np.mean(variances)
    print(f"{n:>6} {e_mean:>12.6f} {e_var:>14.6f} {e_mean - true_mu:>12.6f} {e_var - true_sigma**2:>12.6f}")

print("\nBoth x̄ and s² are unbiased estimators (bias → 0 with more simulations).")

# ── MLE for normal distribution (from scratch)
def normal_log_likelihood(data, mu, sigma):
    """Log-likelihood for normal distribution."""
    n = len(data)
    return -n/2 * np.log(2 * np.pi) - n * np.log(sigma) - np.sum((data - mu)**2) / (2 * sigma**2)


def mle_normal(data):
    """Analytical MLE for normal distribution.
    μ̂ = x̄, σ̂² = (1/n)Σ(xi - x̄)² (note: MLE uses 1/n, not 1/(n-1))
    """
    n = len(data)
    mu_hat = np.mean(data)
    sigma2_hat = np.sum((data - mu_hat)**2) / n  # MLE variance (biased)
    return mu_hat, np.sqrt(sigma2_hat)


# Verify MLE
sample_data = rng.normal(true_mu, true_sigma, 1000)
mu_mle, sigma_mle = mle_normal(sample_data)
print(f"\nMLE estimates (n=1000): μ̂ = {mu_mle:.4f}, σ̂ = {sigma_mle:.4f}")
print(f"True values:           μ = {true_mu:.4f}, σ = {true_sigma:.4f}")

### Interpreting the Point Estimator Results

Look at the output table. The key observations are:

1. **The bias columns are very close to zero at every sample size.** This confirms that both $\bar{X}$ and $s^2$ (with Bessel's correction) are unbiased — the expected value of each estimator equals the true parameter, even for small $n$.

2. **The bias does not *systematically* improve with $n$.** For a truly unbiased estimator, bias is zero in theory; the small numbers you see are just Monte Carlo noise. If we ran more simulations (say 100,000 instead of 5,000), these would get even closer to zero.

3. **The MLE estimates are very close to the true values** with $n = 1000$. For large samples, the distinction between MLE (dividing by $n$) and unbiased (dividing by $n-1$) is negligible: $1000$ vs. $999$ in the denominator.

> **Key Concept: Unbiasedness is a property of the *procedure*, not of any single estimate.** A single sample might give $\bar{X} = 0.12$ when $\mu = 0.10$ — that individual estimate is "wrong" by 0.02. But if we could repeat the procedure millions of times, the average of all those $\bar{X}$ values would be exactly 0.10. *That* is what unbiasedness means.

### A Worked Example: Biased vs. Unbiased Variance

Suppose you have 5 monthly returns: 2%, -1%, 3%, 0%, 1%. The sample mean is $\bar{X} = 1\%$.

**MLE variance** (biased, divides by $n$):
$$\hat{\sigma}^2_{MLE} = \frac{(2-1)^2 + (-1-1)^2 + (3-1)^2 + (0-1)^2 + (1-1)^2}{5} = \frac{1+4+4+1+0}{5} = 2.0$$

**Sample variance** (unbiased, divides by $n-1$):
$$s^2 = \frac{1+4+4+1+0}{4} = 2.5$$

The unbiased estimate is 25% larger! For $n = 5$, the bias correction matters. For $n = 100$, the difference between dividing by 100 and 99 is only 1%.

> **CFA Exam Tip:** The exam almost always uses $n-1$ in the denominator for sample variance and standard deviation. If you see $n$ in the denominator, it is likely the MLE or population variance. Read the question carefully.

---
## 4. Confidence Intervals

A point estimate gives us a single number — but how much should we trust it? A $(1-\alpha)$ **confidence interval** provides a range of values that plausibly contains the true parameter.

### The Z-Interval (Known $\sigma$ or Large $n$)

When we know the population standard deviation $\sigma$ (rare in practice), or when $n$ is large enough that $s \approx \sigma$:

$$\bar{x} \pm z_{\alpha/2} \cdot \frac{\sigma}{\sqrt{n}}$$

where $z_{\alpha/2}$ is the critical value from the standard normal distribution.

| Confidence Level | $\alpha$ | $z_{\alpha/2}$ |
|:----------------:|:--------:|:-------------:|
| 90% | 0.10 | 1.645 |
| 95% | 0.05 | 1.960 |
| 99% | 0.01 | 2.576 |

### The T-Interval (Unknown $\sigma$, Small $n$)

When $\sigma$ is unknown (the usual case in finance) and $n$ is small:

$$\bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

where $t_{\alpha/2, n-1}$ is the critical value from the Student's $t$-distribution with $n-1$ degrees of freedom.

**Why the $t$-distribution?** When we replace $\sigma$ with $s$, we introduce extra uncertainty. The $t$-distribution accounts for this by having *fatter tails* than the normal, especially for small $n$. As $n \to \infty$, $t_{n-1} \to N(0,1)$.

### The Correct Interpretation of Confidence Intervals

This is one of the most commonly misunderstood concepts in all of statistics, and the CFA exam loves to test it.

> **Key Concept:** A 95% confidence interval does **NOT** mean "there is a 95% probability that the true mean is in this interval."

The true mean $\mu$ is a fixed (but unknown) number. It is either in the interval or it is not — there is nothing random about $\mu$. What *is* random is the interval itself, because it depends on the random sample.

**The correct interpretation:** "If we repeated this sampling and estimation procedure many times, 95% of the resulting intervals would contain the true parameter."

Think of it as a statement about the **procedure's reliability**, not about any single interval.

**An analogy:** A factory produces fishing nets. Each net catches a random set of fish. The factory claims "95% of our nets catch the big fish." Once you cast a specific net, the big fish is either caught or not — there is no 95% about it. The 95% describes the *net-making process*, not any individual net.

> **Common Mistake:** This incorrect interpretation — "95% probability the true mean is in this interval" — is called a **Bayesian** interpretation and requires a prior distribution on $\mu$. In the classical (frequentist) framework used in the CFA curriculum, $\mu$ is fixed and the interval is random. The exam will present the incorrect interpretation as a wrong answer choice.

### Z-Interval vs. T-Interval: When to Use Which

| Condition | Use |
|:----------|:----|
| Normal population, $\sigma$ known | Z-interval |
| Normal population, $\sigma$ unknown | T-interval |
| Non-normal population, large $n$ ($\geq 30$), $\sigma$ known | Z-interval (CLT justifies normality of $\bar{X}$) |
| Non-normal population, large $n$ ($\geq 30$), $\sigma$ unknown | T-interval (but Z-interval is also acceptable) |
| Non-normal population, small $n$, $\sigma$ unknown | Neither is reliable — consider nonparametric methods (bootstrap) |

> **CFA Exam Tip:** In practice, you almost always use the t-interval because $\sigma$ is almost never known. When $n \geq 30$, the t and z intervals give virtually identical results because $t_{n-1}$ and $N(0,1)$ converge. The exam may present a scenario with $n = 200$ and ask whether you should use z or t — both are acceptable, but the t-interval is technically more correct.

### Worked Example

A portfolio manager computes the average monthly return of her fund from the last 50 months: $\bar{X} = 0.8\%$ with sample standard deviation $s = 4\%$.

**95% t-interval:**
$$0.8\% \pm t_{0.025, 49} \times \frac{4\%}{\sqrt{50}} = 0.8\% \pm 2.010 \times 0.566\% = 0.8\% \pm 1.137\%$$
$$= [-0.337\%,\ 1.937\%]$$

**Interpretation:** If we repeated this 50-month sampling procedure many times, 95% of the resulting intervals would contain the true mean monthly return. Note that **zero is inside this interval**, which means we cannot conclude (at the 95% level) that the fund's true mean return is positive.

This is the fundamental challenge in finance: returns are noisy (high $\sigma$), so you need a lot of data to detect even moderate effects.### The Most Common Misinterpretation in Statistics

Almost everyone gets this wrong, including many finance professionals:

> **Common Mistake:** "There is a 95% probability that the true mean is between 7.2% and 12.8%." This is **WRONG**.

The correct interpretation:

> **Key Concept:** "If we repeated this sampling procedure many times, 95% of the resulting intervals would contain the true mean." The true mean is a fixed (unknown) constant — it's either in the interval or it's not. There's no probability involved once the interval is computed.

**Analogy:** Imagine throwing horseshoes at a stake. Before you throw, there's a 95% chance your horseshoe will land around the stake. But once it's landed, it either ringed the stake or it didn't. Looking at a landed horseshoe and saying "there's a 95% chance it ringed the stake" doesn't make sense.

### z-Interval vs t-Interval: When to Use Which

| Situation | Use | Why |
|:----------|:----|:----|
| Known $\sigma$, any $n$ | z-interval | Exact critical values from normal distribution |
| Unknown $\sigma$, $n \geq 30$ | Either (z is approximate) | CLT ensures normality of $\bar{X}$; $s \approx \sigma$ |
| Unknown $\sigma$, $n < 30$ | **t-interval** | Extra uncertainty from estimating $\sigma$ widens the interval |


### Building Confidence Intervals from Scratch

Let's implement both z-intervals and t-intervals and see how they compare. We draw a sample of 50 returns from a population with $\mu = 8\%$ and $\sigma = 15\%$.

Pay attention to the difference in width: the t-interval is *wider* than the z-interval because it accounts for the extra uncertainty of estimating $\sigma$ with $s$.

The key difference in the code:
- **Z-interval:** uses `stats.norm.ppf()` (the normal distribution) and the *known* $\sigma$
- **T-interval:** uses `stats.t.ppf()` (the t-distribution) and the *estimated* $s$

For $n = 50$, the critical values are:
- $z_{0.025} = 1.960$
- $t_{0.025, 49} \approx 2.010$

The t critical value is slightly larger, making the interval wider. For $n = 10$, this gap would be much more dramatic: $z_{0.025} = 1.960$ vs. $t_{0.025, 9} = 2.262$.

> **What to expect:** The t-interval will be slightly wider than the z-interval. The difference is small when $n = 50$ (because $t_{49}$ is close to $N(0,1)$), but would be much larger if $n = 10$.

In [ ]:
def z_confidence_interval(data, sigma, alpha=0.05):
    """Z-interval for the mean (known population sigma).
    
    Returns: (lower, upper, point_estimate)
    """
    n = len(data)
    x_bar = np.mean(data)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    margin = z_crit * sigma / np.sqrt(n)
    return x_bar - margin, x_bar + margin, x_bar


def t_confidence_interval(data, alpha=0.05):
    """T-interval for the mean (unknown sigma).
    
    Returns: (lower, upper, point_estimate)
    """
    n = len(data)
    x_bar = np.mean(data)
    s = np.sqrt(np.sum((data - x_bar)**2) / (n - 1))
    t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
    margin = t_crit * s / np.sqrt(n)
    return x_bar - margin, x_bar + margin, x_bar


# ── Example: Confidence interval for mean return
sample_returns = rng.normal(0.08, 0.15, 50)

# Z-interval (pretend we know sigma = 0.15)
lo_z, hi_z, pt_z = z_confidence_interval(sample_returns, sigma=0.15, alpha=0.05)
# T-interval (sigma unknown)
lo_t, hi_t, pt_t = t_confidence_interval(sample_returns, alpha=0.05)

print(f"Sample: n = {len(sample_returns)}, x̄ = {pt_z:.4%}, s = {np.std(sample_returns, ddof=1):.4%}")
print(f"\n95% Confidence Intervals:")
print(f"  Z-interval: [{lo_z:.4%}, {hi_z:.4%}]  width = {(hi_z-lo_z):.4%}")
print(f"  T-interval: [{lo_t:.4%}, {hi_t:.4%}]  width = {(hi_t-lo_t):.4%}")
print(f"\nT-interval is wider because it accounts for the extra uncertainty")
print(f"from estimating σ with s.")

### The Coverage Experiment: Seeing the Correct Interpretation

The most powerful way to understand what a confidence interval truly means is to **build many of them** and see how many capture the true parameter.

Below, we generate 100 independent samples (each of size 30), build a 95% t-interval from each, and visually check how many contain the true mean ($\mu = 8\%$). If our formula and theory are correct, approximately 95 out of 100 intervals should contain $\mu$.

- **Green bars:** intervals that successfully captured the true mean
- **Red bars:** intervals that missed

> **What to expect:** About 95 green bars and 5 red ones. The exact count will vary due to randomness, but it should be close to 95.

In [ ]:
# ── Visualization: Coverage of confidence intervals
# Generate many 95% CIs and check what fraction contain the true mean

true_mu_ci = 0.08
n_intervals = 100
n_sample_ci = 30
alpha_ci = 0.05

fig, ax = plt.subplots(figsize=(12, 8))

contains_count = 0
for i in range(n_intervals):
    sample = rng.normal(true_mu_ci, 0.15, n_sample_ci)
    lo, hi, pt = t_confidence_interval(sample, alpha_ci)
    
    contains = lo <= true_mu_ci <= hi
    contains_count += contains
    color = TERTIARY if contains else SECONDARY
    
    ax.plot([lo, hi], [i, i], color=color, linewidth=1.5, alpha=0.8)
    ax.plot(pt, i, 'o', color=color, markersize=3)

ax.axvline(true_mu_ci, color='black', linewidth=2, linestyle='--', label=f'True μ = {true_mu_ci:.0%}')
ax.set_xlabel('Return')
ax.set_ylabel('Sample Index')
ax.set_title(f'95% Confidence Intervals: {contains_count}/{n_intervals} contain true mean\n'
             f'(Green = contains μ, Red = misses μ)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Coverage rate: {contains_count/n_intervals:.0%} (expected ≈ 95%)")

### Interpreting the Coverage Experiment

Count the red bars (intervals that missed the true mean). You should see roughly 5 out of 100. This is exactly the correct interpretation of a 95% confidence interval in action:

- **Before** we draw a sample, there is a 95% probability that the procedure will produce an interval containing $\mu$.
- **After** we draw a specific sample and compute a specific interval, $\mu$ is either in it or not. We just don't know which.

Look at the red bars. Notice that they are not defective in any obvious way — they are not systematically shorter or off to one side. They simply represent samples where, by random chance, $\bar{X}$ happened to land far from $\mu$, pulling the entire interval to one side.

> **Key Concept:** The 5% of intervals that miss $\mu$ are *not errors in the formula*. They are a feature, not a bug. We *chose* 95% confidence, knowing that 5% of intervals would miss. If we wanted fewer misses, we could use 99% confidence — but that would make every interval wider (less precise). This is the **confidence-precision tradeoff**.

> **Common Mistake:** "My 95% confidence interval for the mean return is [2%, 8%], so there is a 95% chance the true mean is between 2% and 8%." **Wrong.** The true mean is either between 2% and 8%, or it is not. The correct statement: "The procedure I used generates intervals that contain the true mean 95% of the time." This is a statement about the *method's long-run reliability*, not about this particular interval.

### Factors That Affect Confidence Interval Width

The width of a confidence interval is $2 \times z_{\alpha/2} \times \sigma / \sqrt{n}$ (for the z-interval). Three factors control it:

| Factor | Effect on width | Intuition |
|:-------|:---------------|:----------|
| Increase confidence ($1-\alpha$) | Wider | More confidence requires a wider net |
| Increase sample size ($n$) | Narrower (by $\sqrt{n}$) | More data = more precision |
| Increase variability ($\sigma$) | Wider | Noisier data = less precision |

In finance, $\sigma$ is often large (returns are volatile) and $n$ is limited (we only have so many years of data). This is why confidence intervals for financial parameters tend to be frustratingly wide.

---
## 5. Bootstrap Methods

### The Problem: When Formulas Fail

The confidence interval formulas above work well for the *mean*. But what about more complex statistics like:
- The **median** return?
- The **Sharpe ratio**?
- The **Value at Risk (VaR)** at the 5th percentile?
- The **correlation** between two assets?

For these, there is no simple formula like $\bar{x} \pm z_{\alpha/2} \cdot \sigma/\sqrt{n}$. The sampling distributions of these statistics are complex, often non-normal, and depend on the underlying data distribution in intricate ways.

This is where the **bootstrap** comes in.

### The Bootstrap Philosophy: "If You Can't Derive It, Simulate It"

The bootstrap is one of the most elegant ideas in modern statistics. The core insight is brilliantly simple:

1. **Ideal world:** To find the sampling distribution of a statistic, you would draw thousands of samples from the *population* and compute the statistic each time.
2. **Problem:** You only have one sample. You cannot go back and re-sample from the population.
3. **Bootstrap solution:** Treat your sample as if it *were* the population. Resample from it (with replacement) thousands of times. The resulting distribution of the statistic approximates the true sampling distribution.

**Why resampling *with replacement* is crucial.** If we resampled *without* replacement, we would get back the same sample every time (just reordered). Sampling *with* replacement means some observations appear multiple times and others are left out, creating genuine variation from resample to resample.

### The Bootstrap Algorithm

Given observed data $x_1, \ldots, x_n$ and a statistic of interest $\hat{\theta} = g(x_1, \ldots, x_n)$:

1. For $b = 1, \ldots, B$ (typically $B = 10{,}000$):
   - Draw a **bootstrap sample** of size $n$ **with replacement** from the observed data
   - Compute the statistic: $\hat{\theta}_b^* = g(x_1^*, \ldots, x_n^*)$
2. The collection $\{\hat{\theta}_1^*, \ldots, \hat{\theta}_B^*\}$ is the **bootstrap distribution**
3. Use this distribution to compute standard errors, confidence intervals, or bias estimates

### Bootstrap Confidence Interval (Percentile Method)

The simplest bootstrap CI just takes percentiles of the bootstrap distribution:

$$CI_{1-\alpha} = [\hat{\theta}^*_{\alpha/2}, \hat{\theta}^*_{1-\alpha/2}]$$

For a 95% CI, take the 2.5th and 97.5th percentiles of the bootstrap distribution.

### Bootstrap Standard Error

The standard deviation of the bootstrap distribution is the **bootstrap standard error**:

$$\widehat{SE}_{boot} = \sqrt{\frac{1}{B-1}\sum_{b=1}^{B}(\hat{\theta}_b^* - \bar{\hat{\theta}}^*)^2}$$

### Advantages of the Bootstrap

- **No distributional assumptions** — works whether data is normal, skewed, fat-tailed, or anything else
- **Works for any statistic** — mean, median, Sharpe ratio, VaR, correlation, you name it
- **Automatically captures asymmetry** — if the sampling distribution is skewed, the bootstrap CI will be asymmetric (unlike the symmetric $\pm$ z-interval)
- **Particularly useful for financial data**, which is often non-normal

### Limitations

- Requires that the sample is representative of the population (garbage in, garbage out)
- Can break down for very small samples ($n < 15$ or so)
- Computationally intensive (though trivial with modern computers)
- The percentile method can be inaccurate for highly biased estimators (more sophisticated variants like the BCa bootstrap address this)

> **Key Concept:** The bootstrap does not create new information. It cannot overcome a small or biased sample. What it *can* do is extract the maximum information from the sample you have, without requiring you to assume a particular distribution.

> **CFA Exam Tip:** The CFA curriculum introduces bootstrap as a "resampling method." The exam may ask you to describe the procedure or identify its advantages. Key points to remember: (1) sample *with replacement*, (2) repeat many times, (3) use the distribution of the resampled statistics for inference.### The Bootstrap Principle

The bootstrap rests on a brilliant insight:

> **Key Concept:** If the sample is a good representation of the population, then **resampling from the sample** mimics **resampling from the population**. The variability of bootstrap estimates approximates the variability of the estimator itself.

The procedure:
1. Draw $B$ samples of size $n$ **with replacement** from your original data
2. Compute the statistic of interest on each bootstrap sample
3. The distribution of these $B$ statistics IS your estimate of the sampling distribution
4. Use percentiles of this distribution as confidence interval bounds

**Why "with replacement"?** Because sampling with replacement from a sample of size $n$ creates variability that mimics the variability of sampling from the population. Without replacement, you'd just get the same data each time.

> **CFA Exam Tip:** The bootstrap is especially valuable for statistics that lack simple formulas for standard errors — like the **Sharpe ratio**, **VaR**, **median**, or **tracking error**. In practice, risk managers use bootstrap extensively.


### The Bootstrap in Action

Let's apply the bootstrap to a realistic financial scenario. We have 60 months of hedge fund returns that are:
- **Non-normal:** 55 "normal" months from $N(0.8\%, 3\%)$ and 5 crisis months from $N(-10\%, 5\%)$
- **Left-skewed:** the crisis months create a heavy left tail
- **Fat-tailed:** the occasional large losses make the kurtosis much higher than 3

For this kind of data, the standard formulas (which assume normality) can be misleading. The bootstrap handles it gracefully.

We bootstrap three statistics:
1. **Mean return** — for comparison with the analytical formula
2. **Median return** — no simple analytical formula for its standard error
3. **Sharpe ratio** — a complex function of mean and standard deviation; its sampling distribution is notoriously non-normal

> **What to watch for:** The bootstrap CI for the Sharpe ratio may be asymmetric (wider on one side), reflecting the skewness of its sampling distribution. The standard analytical formula would produce a symmetric interval, which is incorrect.

In [ ]:
def bootstrap(data, statistic_fn, B=10_000, alpha=0.05, rng=None):
    """Nonparametric bootstrap.
    
    Parameters
    ----------
    data         : 1D array of observations
    statistic_fn : callable, takes array → scalar
    B            : number of bootstrap resamples
    alpha        : significance level for CI
    rng          : numpy random generator
    
    Returns
    -------
    dict with: estimate, se, ci_lower, ci_upper, bootstrap_dist
    """
    if rng is None:
        rng = np.random.default_rng()
    
    data = np.asarray(data)
    n = len(data)
    
    # Original estimate
    theta_hat = statistic_fn(data)
    
    # Bootstrap resamples
    boot_stats = np.zeros(B)
    for b in range(B):
        boot_sample = data[rng.integers(0, n, size=n)]
        boot_stats[b] = statistic_fn(boot_sample)
    
    # Standard error
    se = np.std(boot_stats, ddof=1)
    
    # Percentile confidence interval
    ci_lower = np.percentile(boot_stats, 100 * alpha / 2)
    ci_upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    
    return {
        'estimate': theta_hat,
        'se': se,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'bootstrap_dist': boot_stats,
    }


# ── Generate realistic hedge fund returns (skewed, fat-tailed)
hf_returns = np.concatenate([
    rng.normal(0.008, 0.03, 55),    # normal months
    rng.normal(-0.10, 0.05, 5),     # crisis months (left tail events)
])
rng.shuffle(hf_returns)

# ── Bootstrap the mean, median, and Sharpe ratio
def sharpe_fn(x, rf_monthly=0.003):
    excess = x - rf_monthly
    return np.mean(excess) / np.std(excess, ddof=1) * np.sqrt(12)  # annualized

statistics = {
    'Mean Return': np.mean,
    'Median Return': np.median,
    'Sharpe Ratio (ann.)': sharpe_fn,
}

print(f"Hedge Fund Monthly Returns (n={len(hf_returns)})\n")
print(f"{'Statistic':<25} {'Estimate':>10} {'Boot SE':>10} {'95% CI':>24}")
print("-" * 72)

boot_results = {}
for name, fn in statistics.items():
    result = bootstrap(hf_returns, fn, B=10_000, rng=rng)
    boot_results[name] = result
    print(f"{name:<25} {result['estimate']:>10.4f} {result['se']:>10.4f} "
          f"[{result['ci_lower']:>10.4f}, {result['ci_upper']:>10.4f}]")

### Interpreting the Bootstrap Results

Several important observations from the output:

1. **Mean return:** The bootstrap SE and CI can be compared with the analytical formula ($s/\sqrt{n}$). They should be very similar, confirming that the bootstrap works correctly for simple statistics.

2. **Median return:** Notice the bootstrap SE — there is no simple closed-form for this. Without the bootstrap, you would need to make strong distributional assumptions (e.g., assume normality) to get a confidence interval. The bootstrap gives it to you for free.

3. **Sharpe ratio:** This is where the bootstrap really shines. The Sharpe ratio is a ratio of two random variables (mean excess return / standard deviation), making its sampling distribution complex. The bootstrap CI may be asymmetric — check whether the distance from the estimate to the lower bound differs from the distance to the upper bound.

> **Key Concept:** For the Sharpe ratio specifically, the analytical formula for its standard error (Lo, 2002) assumes normality of returns. When returns are skewed or fat-tailed — as they almost always are for hedge funds — the bootstrap provides a more honest assessment of uncertainty.

### Visualising the Bootstrap Distributions

The histograms below show the bootstrap distributions for each statistic. Each histogram represents 10,000 bootstrap resamples.

Look for:
- **Symmetry:** The mean's bootstrap distribution should be roughly symmetric (CLT again). The median and Sharpe ratio may be noticeably skewed.
- **Width:** Wider distributions indicate more uncertainty.
- **The CI bounds** (red dotted lines): these cut off the extreme 2.5% on each side.

In [ ]:
# ── Visualization: Bootstrap distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [PRIMARY, SECONDARY, TERTIARY]

for ax, (name, result), color in zip(axes, boot_results.items(), colors):
    ax.hist(result['bootstrap_dist'], bins=60, density=True, alpha=0.7, 
            color=color, edgecolor='white')
    ax.axvline(result['estimate'], color='black', linewidth=2, linestyle='--', label='Estimate')
    ax.axvline(result['ci_lower'], color='darkred', linewidth=1.5, linestyle=':', label='95% CI')
    ax.axvline(result['ci_upper'], color='darkred', linewidth=1.5, linestyle=':')
    ax.set_title(name)
    ax.set_xlabel('Value')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Density')
plt.suptitle('Bootstrap Distributions (B=10,000)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Reading the Bootstrap Histograms

Each histogram tells a different story about estimation uncertainty:

- **Mean Return:** Approximately bell-shaped (the CLT is at work even within the bootstrap). The CI bounds are roughly symmetric around the estimate. This confirms that for the mean, the standard analytical formula works fine — the bootstrap and the formula agree.

- **Median Return:** May appear slightly skewed or multi-modal. The median is a "discrete" statistic in the sense that it jumps between data points, which can create a lumpy bootstrap distribution. This is perfectly normal and is one reason why the median's standard error is hard to compute analytically.

- **Sharpe Ratio:** Often the most interesting distribution. If the hedge fund data has significant negative skewness (from crisis months), the bootstrap distribution of the Sharpe ratio may show a noticeable left tail — reflecting the risk that a few bad months can dramatically reduce the realized Sharpe ratio.

### When to Use Bootstrap vs. Analytical Formulas

| Statistic | Analytical formula available? | Bootstrap recommended? |
|:----------|:----------------------------|:----------------------|
| Mean | Yes ($s/\sqrt{n}$) | Optional — both give similar results |
| Median | Not simple | Yes — bootstrap is the standard approach |
| Sharpe ratio | Approximate (assumes normality) | Yes — especially for non-normal returns |
| VaR (percentile) | Not straightforward | Yes — one of the main use cases |
| Correlation | Fisher z-transform | Bootstrap captures non-normality better |
| Regression coefficients | Yes (OLS formulas) | Useful when errors are non-normal |

The general rule: **use analytical formulas when their assumptions are met; use bootstrap when they are not** (or when you are unsure). For financial data, the bootstrap is almost always a safer choice.

> **Common Mistake:** Some practitioners report the Sharpe ratio as a single number (e.g., "our fund has a Sharpe of 1.2") without any uncertainty estimate. The bootstrap shows just how imprecise this number can be. A Sharpe ratio of 1.2 with a bootstrap 95% CI of [0.4, 2.1] is very different from one with a CI of [1.0, 1.4]. Always report uncertainty alongside point estimates.

> **CFA Exam Tip:** The exam may present a scenario where you need to compute a confidence interval for a statistic other than the mean (e.g., median, variance ratio). If no analytical formula is given, the bootstrap is the go-to method. Remember the key steps: resample with replacement, compute the statistic each time, take percentiles.

---
## 6. Sample Size Determination

One of the most practical questions in applied statistics is: **how many observations do I need?** Before launching a study, an analyst must determine the minimum sample size required to achieve a desired level of precision.

### Approach 1: Based on Margin of Error

If you want a confidence interval with a specified margin of error $E$ at confidence level $1-\alpha$, you need:

$$n = \left(\frac{z_{\alpha/2} \cdot \sigma}{E}\right)^2$$

**Derivation:** Start from the CI formula: $E = z_{\alpha/2} \cdot \sigma / \sqrt{n}$. Solve for $n$:
$$\sqrt{n} = \frac{z_{\alpha/2} \cdot \sigma}{E} \implies n = \left(\frac{z_{\alpha/2} \cdot \sigma}{E}\right)^2$$

**The catch:** You need to know $\sigma$ before collecting data! Common approaches:
- Use $\sigma$ from a pilot study or prior research
- Use a conservative (high) estimate of $\sigma$
- For proportions, use $p = 0.5$ (maximum variance)

### Worked Example: Estimating Mean Return

A research analyst wants to estimate the average monthly return of emerging market bonds within a margin of error of $E = 0.5\%$ at 95% confidence. Historical data suggests monthly volatility is approximately $\sigma = 3\%$.

$$n = \left(\frac{1.96 \times 0.03}{0.005}\right)^2 = \left(\frac{0.0588}{0.005}\right)^2 = (11.76)^2 = 138.3 \implies n = 139$$

The analyst needs at least **139 months** (about 11.5 years) of data. This is a lot! And it highlights why financial estimation is so challenging: returns are noisy relative to their means.

> **CFA Exam Tip:** Always round the sample size UP to the next whole number. You cannot observe 138.3 months of data. Rounding down would give you a margin of error slightly wider than desired.

### Approach 2: Power Analysis

Power analysis determines the sample size needed to *detect an effect* (reject a false null hypothesis) with a specified probability.

**Key concepts:**
- **Significance level ($\alpha$):** Probability of rejecting $H_0$ when it is true (Type I error). Typically 5%.
- **Power ($1 - \beta$):** Probability of rejecting $H_0$ when it is false (correctly detecting a real effect). Typically 80% or 90%.
- **Effect size ($\delta$):** The minimum difference you want to detect.

For a one-sided z-test detecting a difference $\delta = \mu_1 - \mu_0$:

$$n = \left(\frac{(z_{\alpha} + z_{\beta}) \cdot \sigma}{\delta}\right)^2$$

where $z_{\alpha}$ is the critical value for significance and $z_{\beta}$ is the critical value for power.

### Why Power Matters in Finance

Consider a hedge fund that claims to generate 1% monthly alpha. You want to test this claim.

- $H_0: \mu = 0$ (no alpha)
- $H_1: \mu = 0.01$ (1% monthly alpha)
- Monthly $\sigma = 5\%$

For 80% power with 5% significance:
$$n = \left(\frac{(1.645 + 0.842) \times 0.05}{0.01}\right)^2 = \left(\frac{0.1244}{0.01}\right)^2 = (12.44)^2 = 154.7 \implies n = 155$$

You need **155 months** (nearly 13 years) of data to have an 80% chance of detecting 1% monthly alpha! This explains why:
- Short track records are nearly useless for statistical evaluation
- The finance industry relies heavily on qualitative judgment alongside quantitative evidence
- Many "statistically insignificant" strategies may actually be profitable but lack sufficient data

> **Key Concept:** The required sample size is proportional to $(\sigma/\delta)^2$. When the signal-to-noise ratio is low (small $\delta$, large $\sigma$) — as it typically is in finance — you need enormous samples. This is the fundamental statistical challenge of active management.

> **Common Mistake:** Analysts sometimes conclude "this strategy doesn't work" when they fail to reject $H_0$. But if the study was underpowered (too few observations), failure to reject simply means the data was *inconclusive*, not that the effect doesn't exist. Always check power before interpreting a non-significant result.
### The Power of a Test

**Statistical power** = probability of correctly rejecting a false null hypothesis = $1 - \beta$ (where $\beta$ is the Type II error rate).

Power depends on four things:
1. **Sample size** ($n$) — larger $n$ → more power
2. **Effect size** — larger true difference → easier to detect
3. **Significance level** ($\alpha$) — higher $\alpha$ → more power (but more Type I errors)
4. **Population variability** ($\sigma$) — less noise → easier to detect signals

> **CFA Exam Tip:** Increasing $\alpha$ from 1% to 5% makes it easier to reject $H_0$ (more power), but also increases the chance of a false positive. There is always a trade-off between Type I and Type II errors.


### Computing Required Sample Sizes

The code below implements both approaches and generates tables showing how required sample sizes change with different parameters.

**Table 1** shows the sample size needed for various margins of error at 95% and 99% confidence, assuming monthly $\sigma = 5\%$.

**Table 2** shows the power analysis results: how many months of data you need to detect various levels of monthly alpha with 80% or 90% power.

> **What to watch for:** Notice how rapidly $n$ grows as the margin of error shrinks or the desired effect size decreases. Going from $E = 2\%$ to $E = 0.2\%$ requires *100x* more data (because $n$ scales as $1/E^2$).

In [ ]:
def required_sample_size(sigma, margin_of_error, alpha=0.05):
    """Compute required n for a given margin of error."""
    z = stats.norm.ppf(1 - alpha / 2)
    n = (z * sigma / margin_of_error) ** 2
    return int(np.ceil(n))


def power_sample_size(sigma, delta, alpha=0.05, power=0.80):
    """Compute required n for desired power (one-sided test).
    
    Parameters
    ----------
    sigma : population std
    delta : minimum detectable effect (mu_1 - mu_0)
    alpha : significance level
    power : desired power (1 - beta)
    """
    z_alpha = stats.norm.ppf(1 - alpha)
    z_beta = stats.norm.ppf(power)
    n = ((z_alpha + z_beta) * sigma / delta) ** 2
    return int(np.ceil(n))


# ── Example 1: How many months of data to estimate mean return within 1%?
sigma_monthly = 0.05  # 5% monthly std
margins = [0.02, 0.01, 0.005, 0.002]

print("Required Sample Size for Mean Return Estimation")
print(f"Monthly σ = {sigma_monthly:.0%}\n")
print(f"{'Margin of Error':>16} {'n (95% CI)':>12} {'n (99% CI)':>12}")
print("-" * 42)
for E in margins:
    n_95 = required_sample_size(sigma_monthly, E, alpha=0.05)
    n_99 = required_sample_size(sigma_monthly, E, alpha=0.01)
    print(f"{E:>16.2%} {n_95:>12,} {n_99:>12,}")

# ── Example 2: Power analysis for detecting alpha
print(f"\nPower Analysis: Detecting Positive Alpha")
print(f"Monthly σ = {sigma_monthly:.0%}, α = 5%\n")
print(f"{'Min Alpha (monthly)':>20} {'n (power=80%)':>14} {'n (power=90%)':>14}")
print("-" * 50)
for delta in [0.005, 0.01, 0.02, 0.03]:
    n_80 = power_sample_size(sigma_monthly, delta, power=0.80)
    n_90 = power_sample_size(sigma_monthly, delta, power=0.90)
    print(f"{delta:>20.2%} {n_80:>14,} {n_90:>14,}")

### Interpreting the Sample Size Tables

**Table 1 (Margin of Error):**

The numbers are striking. To estimate mean monthly return within $\pm 0.5\%$ at 95% confidence (a seemingly modest requirement), you need nearly 400 months — over 30 years of data! To get within $\pm 0.2\%$, you need over 2,400 months — roughly 200 years. This is clearly impractical, which is why financial estimates always come with substantial uncertainty.

**Table 2 (Power Analysis):**

To detect a monthly alpha of 0.5% (about 6% annualised — a very respectable return) with 80% power, you need over 600 months (50+ years). Even for a stellar monthly alpha of 3%, you need around 17-22 months.

The practical implication: **most fund track records are far too short to statistically distinguish skill from luck.** A fund with 3 years of data (36 months) simply does not have enough observations to provide reliable evidence of alpha, unless the alpha is very large.

### Putting It in Perspective: Real-World Track Records

| Fund tenure | Months of data | What you can detect (80% power, 5% significance) |
|:------------|:--------------:|:--------------------------------------------------|
| 1 year | 12 | Only extremely large alpha (> 4.5% monthly) |
| 3 years | 36 | Large alpha (> 2.5% monthly, or ~30% annually) |
| 5 years | 60 | Moderate alpha (~2% monthly, or ~24% annually) |
| 10 years | 120 | Meaningful alpha (~1.4% monthly, or ~17% annually) |
| 20 years | 240 | Modest alpha (~1% monthly, or ~12% annually) |
| 50 years | 600 | Small alpha (~0.6% monthly, or ~7% annually) |

These numbers are sobering. Warren Buffett's 60+ year track record is one of the few that provides truly compelling *statistical* evidence of skill. For most fund managers with 3-5 year track records, statistical tests are nearly powerless.

> **Key Concept:** The formula $n = (z \cdot \sigma / E)^2$ reveals that required sample size is *inversely proportional to the square* of the margin of error. Halving the margin of error quadruples the required sample size. This is the $\sqrt{n}$ law working in reverse.

> **CFA Exam Tip:** Sample size determination questions are straightforward formula applications. The exam will give you $\sigma$, $E$, and $\alpha$, and ask you to compute $n$. Remember: (1) use the formula, (2) always round UP. A common mistake is rounding down or not squaring the result.

### Visualising Power Curves

The plots below show two complementary views of the power analysis:

**Left panel:** Power as a function of sample size for a fixed effect size (1% monthly alpha). Each curve corresponds to a different volatility level. Higher volatility means you need more data to achieve the same power — the signal is harder to detect in noisier data.

**Right panel:** Required sample size as a function of effect size for fixed power levels. This shows the "diminishing difficulty" of detecting larger effects — detecting 5% alpha is much easier than detecting 0.5% alpha.

In [ ]:
# ── Visualization: Power curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Power as function of n for fixed effect size
n_range = np.arange(10, 501)
delta_fixed = 0.01  # 1% monthly alpha

for sigma_val, color, label in [(0.03, PRIMARY, 'σ=3%'), 
                                  (0.05, SECONDARY, 'σ=5%'),
                                  (0.08, TERTIARY, 'σ=8%')]:
    powers = []
    for n in n_range:
        # Power = P(reject | H1 true) = Phi(delta*sqrt(n)/sigma - z_alpha)
        z_alpha = stats.norm.ppf(0.95)
        power = stats.norm.cdf(delta_fixed * np.sqrt(n) / sigma_val - z_alpha)
        powers.append(power)
    axes[0].plot(n_range, powers, color=color, linewidth=2, label=label)

axes[0].axhline(0.80, color='gray', linestyle='--', alpha=0.7, label='80% power')
axes[0].set_xlabel('Sample Size (n)')
axes[0].set_ylabel('Power')
axes[0].set_title(f'Power Curves (detecting δ = {delta_fixed:.0%} monthly alpha)')
axes[0].legend()

# Right: Required n as function of effect size
deltas = np.linspace(0.003, 0.05, 100)
for power_val, color, label in [(0.80, PRIMARY, '80% power'),
                                  (0.90, SECONDARY, '90% power')]:
    ns = [power_sample_size(0.05, d, power=power_val) for d in deltas]
    axes[1].plot(deltas * 100, ns, color=color, linewidth=2, label=label)

axes[1].set_xlabel('Effect Size (Monthly Alpha, %)')
axes[1].set_ylabel('Required Sample Size')
axes[1].set_title('Sample Size vs Effect Size (σ = 5%)')
axes[1].set_ylim(0, 2000)
axes[1].legend()

plt.tight_layout()
plt.show()

### Reading the Power Curves

**Left panel observations:**

- The **blue curve** ($\sigma = 3\%$, low volatility) reaches 80% power fastest — around $n \approx 55$ months. This is a low-volatility strategy where the signal (1% monthly alpha) is relatively easy to detect.
- The **orange curve** ($\sigma = 5\%$, moderate volatility) needs about $n \approx 155$ months — around 13 years. This is typical for most equity strategies.
- The **green curve** ($\sigma = 8\%$, high volatility) needs about $n \approx 400$ months — over 33 years! For a highly volatile strategy, detecting 1% monthly alpha is extremely difficult.

**Right panel observations:**

- The curves are steeply declining on the left (small effect sizes require enormous samples) and flat on the right (large effect sizes are easy to detect).
- Detecting 1% monthly alpha ($x$-axis = 1.0) with 80% power requires about 155 observations.
- Detecting 3% monthly alpha requires only about 17 observations.
- The 90% power curve is everywhere above the 80% power curve — higher power demands more data.

> **Key Concept: The Signal-to-Noise Ratio.** The ratio $\delta / \sigma$ (effect size divided by noise) is the fundamental determinant of how much data you need. In finance, $\delta$ (the alpha or return premium) is typically small, while $\sigma$ (volatility) is large. This low signal-to-noise ratio is why financial inference is so challenging — and why long track records are so valuable.

> **Common Mistake:** Reporting that a strategy "doesn't work" based on 3 years of data with $p > 0.05$. A quick power calculation would show that 3 years provides less than 20% power to detect a 1% monthly alpha. The study was simply too short to conclude anything meaningful.

---

## Summary: Key Takeaways

| Topic | Key Formula / Idea | Financial Implication |
|:------|:-------------------|:---------------------|
| **Sampling methods** | Stratified > SRS for structured populations | Index fund construction uses stratified sampling |
| **CLT** | $\bar{X} \approx N(\mu, \sigma^2/n)$ for large $n$ | Justifies normal-based tests even for non-normal returns |
| **Standard error** | $SE = \sigma / \sqrt{n}$ | Precision grows slowly with data (diminishing returns) |
| **Point estimators** | Unbiased, efficient, consistent | Use $n-1$ for sample variance (Bessel's correction) |
| **Confidence intervals** | $\bar{x} \pm t_{\alpha/2} \cdot s/\sqrt{n}$ | Always report uncertainty with your estimates |
| **CI interpretation** | 95% of intervals contain $\mu$ | NOT "95% probability $\mu$ is in this interval" |
| **Bootstrap** | Resample with replacement, $B \geq 10{,}000$ | Essential for Sharpe ratio, VaR, and other complex statistics |
| **Sample size** | $n = (z \cdot \sigma / E)^2$ | Detecting alpha requires decades of data |

### Common Exam Pitfalls to Avoid

1. **Confusing parameter and statistic.** $\mu$ is the parameter (fixed, unknown). $\bar{X}$ is the statistic (known, random).
2. **Wrong CI interpretation.** "95% probability the true mean is in the interval" is WRONG. Correct: "95% of intervals from repeated sampling would contain the true mean."
3. **Forgetting Bessel's correction.** Sample variance divides by $n-1$, not $n$.
4. **Not rounding sample size up.** Always ceil the result of the sample size formula.
5. **Ignoring sampling bias.** Survivorship bias, look-ahead bias, and time-period bias can invalidate results regardless of sample size.
6. **Concluding "no effect" from low power.** Failure to reject $H_0$ with a small sample means the data was *inconclusive*, not that there is no effect.

### The Unifying Theme

Every topic in this chapter is connected by a single thread: **uncertainty is quantifiable**. We cannot eliminate uncertainty from financial estimates — returns are inherently noisy and our samples are limited. But we *can* measure it precisely, communicate it honestly, and make decisions that account for it. That is what sampling and estimation theory is for.

---
## References

1. **CFA Institute**, *CFA Program Curriculum Level I*, Quantitative Methods: Sampling and Estimation.
2. **DeFusco, R., McLeavey, D., Pinto, J., & Runkle, D.**, *Quantitative Investment Analysis*, 3rd ed., CFA Institute/Wiley, 2015.
3. **Efron, B. & Tibshirani, R.**, *An Introduction to the Bootstrap*, Chapman & Hall, 1993.
4. **Wasserman, L.**, *All of Statistics*, Springer, 2004.
5. **Lo, A.**, "The Statistics of Sharpe Ratios," *Financial Analysts Journal*, 58(4), 2002.### CFA Level 1 Learning Outcome Alignment

- LOS: Compare and contrast simple random, stratified, and systematic sampling
- LOS: Explain the central limit theorem and its importance
- LOS: Calculate and interpret the standard error of the sample mean
- LOS: Distinguish between a point estimate and a confidence interval
- LOS: Describe the properties of a good estimator (unbiasedness, efficiency, consistency)
- LOS: Calculate and interpret a confidence interval for a population mean
